In [2]:
import cv2
import numpy as np


# ============================================================
#                       工具函数
# ============================================================

def create_line_kernel(angle_deg, length, thickness=1):
    """生成指定角度、长度和粗细的线状结构元素"""
    kernel = cv2.getStructuringElement(cv2.MORPH_RECT, (length, thickness))
    center = (length // 2, thickness // 2)
    rot_mat = cv2.getRotationMatrix2D(center, angle_deg, 1.0)
    cos = np.abs(rot_mat[0, 0])
    sin = np.abs(rot_mat[0, 1])
    new_width = int(length * cos + thickness * sin)
    new_height = int(length * sin + thickness * cos)
    rot_mat[0, 2] += new_width / 2 - center[0]
    rot_mat[1, 2] += new_height / 2 - center[1]
    kernel_rotated = cv2.warpAffine(kernel.astype(np.float32), rot_mat,
                                    (new_width, new_height),
                                    flags=cv2.INTER_NEAREST)
    _, kernel_rotated = cv2.threshold(kernel_rotated, 0.5, 1, cv2.THRESH_BINARY)
    return kernel_rotated.astype(np.uint8)


def directional_blackhat(gray, angles=(0, 30, 60, 90, 120, 150), length=15, thickness=1):
    """多方向黑帽运算，提取毛发响应图"""
    blackhat_max = np.zeros_like(gray, dtype=np.float32)
    for ang in angles:
        kernel = create_line_kernel(ang, length, thickness)
        closed = cv2.morphologyEx(gray, cv2.MORPH_CLOSE, kernel)
        blackhat = np.maximum(closed.astype(np.float32) - gray.astype(np.float32), 0)
        blackhat_max = np.maximum(blackhat_max, blackhat)
    cv2.normalize(blackhat_max, blackhat_max, 0, 255, cv2.NORM_MINMAX)
    return blackhat_max.astype(np.uint8)


def coherence_map(gray, window_size=9):
    """计算局部方向相干性图"""
    Ix = cv2.Sobel(gray, cv2.CV_32F, 1, 0, ksize=3)
    Iy = cv2.Sobel(gray, cv2.CV_32F, 0, 1, ksize=3)
    Ixx, Iyy, Ixy = Ix * Ix, Iy * Iy, Ix * Iy
    sigma = window_size / 6.0
    Sxx = cv2.GaussianBlur(Ixx, (0, 0), sigma)
    Syy = cv2.GaussianBlur(Iyy, (0, 0), sigma)
    Sxy = cv2.GaussianBlur(Ixy, (0, 0), sigma)
    trace = Sxx + Syy
    discriminant = np.sqrt(np.maximum((Sxx - Syy) ** 2 + 4 * Sxy * Sxy, 0))
    lambda1 = (trace + discriminant) / 2.0
    lambda2 = (trace - discriminant) / 2.0
    denom = lambda1 + lambda2
    coh = np.zeros_like(gray, dtype=np.float32)
    mask_denom = denom > 1e-6
    coh[mask_denom] = ((lambda1[mask_denom] - lambda2[mask_denom]) / denom[mask_denom]) ** 2
    return coh


def multi_scale_coherence(gray, window_sizes=(5, 15, 25)):
    """多尺度相干性"""
    coh_max = np.zeros_like(gray, dtype=np.float32)
    for ws in window_sizes:
        coh = coherence_map(gray, window_size=ws)
        coh_max = np.maximum(coh_max, coh)
    return coh_max


def clahe_lab(img, clip_limit=2.0, tile_size=8):
    """LAB 空间 CLAHE 对比度增强"""
    lab = cv2.cvtColor(img, cv2.COLOR_BGR2LAB)
    l, a, b = cv2.split(lab)
    clahe = cv2.createCLAHE(clipLimit=clip_limit, tileGridSize=(tile_size, tile_size))
    l_eq = clahe.apply(l)
    return cv2.cvtColor(cv2.merge([l_eq, a, b]), cv2.COLOR_LAB2BGR)


# ============================================================
#                    毛发检测与去除
# ============================================================

def detect_hair_mask(red_channel,
                     angles=(0, 30, 60, 90, 120, 150),
                     length=15,
                     thickness=1,
                     coh_window_sizes=(5, 15, 25),
                     blackhat_blur_ksize=(3, 3),
                     binary_thresh=12,
                     morph_open_ksize=(3, 3),
                     morph_close_ksize=(10, 10)):
    """
    检测毛发并返回二值 mask

    参数
    ----
    red_channel : 红通道灰度图
    angles : 黑帽方向列表
    length : 线核长度
    thickness : 线核粗细
    coh_window_sizes : 多尺度相干性窗口大小
    blackhat_blur_ksize : 黑帽图高斯模糊核
    binary_thresh : 加权黑帽二值化阈值
    morph_open_ksize : 开运算核大小（去噪点）
    morph_close_ksize : 闭运算核大小（连断裂）

    返回
    ----
    mask : 毛发二值图
    mask_ratio : 毛发占比
    """
    # 多方向黑帽
    blackhat = directional_blackhat(red_channel, angles, length, thickness)

    # 高斯平滑
    bhg = cv2.GaussianBlur(blackhat, blackhat_blur_ksize, 0)

    # 多尺度相干性
    coh = multi_scale_coherence(red_channel, coh_window_sizes)

    # 加权黑帽
    weighted = bhg.astype(np.float32) * coh

    # 二值化
    _, mask = cv2.threshold(weighted.astype(np.uint8), binary_thresh, 255, cv2.THRESH_BINARY)

    # 形态学后处理
    kernel_open = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, morph_open_ksize)
    kernel_close = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, morph_close_ksize)
    mask = cv2.morphologyEx(mask, cv2.MORPH_OPEN, kernel_open)
    mask = cv2.morphologyEx(mask, cv2.MORPH_CLOSE, kernel_close)

    mask_ratio = np.sum(mask > 0) / mask.size

    return mask, mask_ratio


def remove_hair(img, mask, inpaint_radius=8, max_mask_ratio=0.35):
    """
    用 inpaint 去除毛发

    返回
    ----
    dst : 去毛发后的图像
    is_valid : 图像是否合格（毛发占比未超标）
    mask_ratio : 毛发占比
    """
    mask_ratio = np.sum(mask > 0) / mask.size
    
    if mask_ratio > max_mask_ratio:
        #  print(f"Mask ratio {mask_ratio:.4f} > {max_mask_ratio}, image rejected.")
        is_valid = False
    else:
        #  print(f"Mask ratio {mask_ratio:.4f} <= {max_mask_ratio}, image accepted.")
        is_valid = True
    
    # 不管是否合格，都做修复（不合格的图至少尝试修复）
    dst = cv2.inpaint(img, mask, inpaint_radius, cv2.INPAINT_TELEA)
    
    return dst, is_valid, mask_ratio




# ============================================================
#                    病灶分割
# ============================================================

def segment_lesion_s_channel(img_bgr,
                              morph_open_ksize=(3, 3),
                              morph_close_ksize=(6, 6)):
    """
    自适应分割病灶：
    - 黑色像素多 → 亮度通道 L + 反色 Otsu（黑色病灶）
    - 黑色像素少 → 饱和度 S 通道（红色/彩色病灶）
    
    返回
    ----
    mask : 病灶二值图
    threshold : Otsu 阈值
    """
    # ========================
    # 第一步：计算黑色像素占比，判断病灶类型
    # ========================
    gray = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2GRAY)
    height, width = gray.shape
    total_pixels = height * width

    # 亮度 < 60 视为黑色病灶区域
    black_pixels = np.sum(gray < 60)
    black_ratio = black_pixels / total_pixels


    # ========================
    # 第二步：自适应选择分割通道
    # ========================
    mask = None
    ret = 0

    # 情况1：黑色病灶 → 使用亮度 L 通道 + 反二值化
    if black_ratio > 0.025:
        lab = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2LAB)
        l_channel = lab[:, :, 0]
        ret, mask = cv2.threshold(l_channel, 0, 255, cv2.THRESH_BINARY_INV + cv2.THRESH_OTSU)

    # 情况2：红色/彩色病灶 → 使用 S 通道
    else:
        hsv = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2HSV)
        s_channel = hsv[:, :, 1]
        ret, mask = cv2.threshold(s_channel, 0, 255, cv2.THRESH_BINARY + cv2.THRESH_OTSU)

    # ========================
    # 形态学后处理（通用）
    # ========================
    kernel_open = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, morph_open_ksize)
    kernel_close = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, morph_close_ksize)
    mask = cv2.morphologyEx(mask, cv2.MORPH_OPEN, kernel_open)
    mask = cv2.morphologyEx(mask, cv2.MORPH_CLOSE, kernel_close)


    # ========================
    # 【核心升级】取前3大连通域 → 选最靠近中心的
    # ========================
    contours, _ = cv2.findContours(mask.copy(), cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
    final_mask = np.zeros_like(mask)

    if contours:
        # 1. 按面积从大到小排序
        contours = sorted(contours, key=cv2.contourArea, reverse=True)
        
        # 2. 只保留前 3 个最大的
        top_contours = contours[:3]

        best_contour = None
        min_distance = float('inf')

        # 3. 遍历前3个，找【距离图像中心最近】的轮廓
        for cnt in top_contours:
            # 计算轮廓的外接矩形中心
            M = cv2.moments(cnt)
            if M["m00"] == 0:
                continue
            cx = int(M["m10"] / M["m00"])
            cy = int(M["m01"] / M["m00"])

            # 计算到图像中心的欧氏距离
            dx = cx - img_bgr.shape[1] / 2
            dy = cy - img_bgr.shape[0] / 2
            distance = (dx**2 + dy**2) ** 0.5

            # 保留距离最小的
            if distance < min_distance:
                min_distance = distance
                best_contour = cnt

        # 4. 画出最优轮廓（最靠近中心）
        if best_contour is not None:
            cv2.fillPoly(final_mask, [best_contour], 255)
        else:
            final_mask = mask

    return final_mask, ret



# ============================================================
#                    可视化工具
# ============================================================

def draw_contour_overlay(img, mask, color=(0, 255, 0), thickness=2):
    """在图像上叠加 mask 轮廓"""
    overlay = img.copy()
    contours, _ = cv2.findContours(mask.copy(), cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
    cv2.drawContours(overlay, contours, -1, color, thickness)
    return overlay


# ============================================================
#                       完整流水线
# ============================================================

def process_pipeline(image_path,verbose=True,
                     # 毛发检测参数
                     angles=(0, 30, 60, 90, 120, 150),
                     length=15,
                     thickness=1,
                     coh_window_sizes=(15, 25),
                     binary_thresh=12,
                     hair_morph_open=(3, 3),
                     hair_morph_close=(10, 10),
                     inpaint_radius=8,
                     max_mask_ratio=0.35,
                     # CLAHE 参数
                     clip_limit=2.0,
                     tile_size=8,
                     # 病灶分割参数
                     lesion_morph_open=(3, 3),
                     lesion_morph_close=(6, 6)):
    """
    完整处理流水线：毛发去除 → CLAHE 增强 → 病灶分割

    返回
    ----
    img_original : 原始图像
    img_hair_removed : 去毛发后
    img_enhanced : CLAHE 增强后
    hair_mask : 毛发二值图
    hair_ratio : 毛发占比
    is_valid : 图像是否合格（True=可用于后续处理）
    lesion_mask : 病灶二值图
    lesion_threshold : 病灶Otsu阈值
    overlay : 病灶轮廓叠加图
    """
    # 读取
    img = cv2.imread(image_path, cv2.IMREAD_COLOR)

    # 红通道
    red_channel = img[:, :, 2]

    # 毛发检测
    hair_mask, hair_ratio = detect_hair_mask(
        red_channel,
        angles=angles,
        length=length,
        thickness=thickness,
        coh_window_sizes=coh_window_sizes,
        binary_thresh=binary_thresh,
        morph_open_ksize=hair_morph_open,
        morph_close_ksize=hair_morph_close
    )
    if verbose:
        print(f"Hair mask ratio: {hair_ratio:.4f}")

    # 毛发去除（返回 is_valid 状态）
    img_hair_removed, is_valid, hair_ratio = remove_hair(
        img, hair_mask, inpaint_radius, max_mask_ratio
    )

    # CLAHE 增强
    img_enhanced = clahe_lab(img_hair_removed, clip_limit, tile_size)

    # 病灶分割
    lesion_mask, lesion_thresh = segment_lesion_s_channel(
        img_enhanced,
        morph_open_ksize=lesion_morph_open,
        morph_close_ksize=lesion_morph_close
    )
    if verbose:
        print(f"Lesion Otsu threshold: {lesion_thresh}")

    # 轮廓叠加
    overlay = draw_contour_overlay(img_enhanced, lesion_mask)

    return {
        'original': img,
        'hair_removed': img_hair_removed,
        'enhanced': img_enhanced,
        'hair_mask': hair_mask,
        'hair_ratio': hair_ratio,
        'is_valid': is_valid,
        'lesion_mask': lesion_mask,
        'lesion_threshold': lesion_thresh,
        'overlay': overlay,
    }




In [6]:
import os
import cv2
import numpy as np
import pandas as pd
import time

# ============================================================
#                        配置参数
# ============================================================

CSV_PATH = "metadata.csv"
IMG_DIR = "data/"
OUTPUT_DIR = "output/nv/"
DX_FILTER = "nv"

ORIGIN_DIR = os.path.join(OUTPUT_DIR, "origin")
ENHANCED_DIR = os.path.join(OUTPUT_DIR, "enhanced")
MASKS_DIR = os.path.join(OUTPUT_DIR, "masks")

# ============================================================
#                        初始化
# ============================================================

for d in [ORIGIN_DIR, ENHANCED_DIR, MASKS_DIR]:
    os.makedirs(d, exist_ok=True)

df = pd.read_csv(CSV_PATH)
df_subset = df[df['dx'] == DX_FILTER]
total = len(df_subset)

# ============================================================
#                        批量处理
# ============================================================

records = []
accepted = 0
rejected = 0
start_time = time.time()

for idx, row in df_subset.iterrows():
    image_id = row['image_id']
    img_path = os.path.join(IMG_DIR, f"{image_id}.jpg")

    result = process_pipeline(img_path, verbose=False)

    if result is None:
        continue

    # 保存
    cv2.imwrite(os.path.join(ORIGIN_DIR, f"{image_id}.jpg"), result['original'])
    cv2.imwrite(os.path.join(ENHANCED_DIR, f"{image_id}.jpg"), result['enhanced'])
    cv2.imwrite(os.path.join(MASKS_DIR, f"{image_id}.png"), result['lesion_mask'])

    # 记录
    records.append({
        **row.to_dict(),
        'hair_ratio': round(result['hair_ratio'], 3),  # ← 加 round(..., 3)
        'is_valid': result['is_valid'],
    })


    if result['is_valid']:
        accepted += 1
    else:
        rejected += 1

    # 动态刷新进度
    print(f"\r正在处理: {len(records)}/{total}", end='')

# ============================================================
#                        保存 & 报告
# ============================================================

df_out = pd.DataFrame(records)
df_out.to_csv(os.path.join(OUTPUT_DIR, "metadata.csv"), index=False)

total_time = time.time() - start_time
print(f"\n\n{'='*50}")
print(f"筛选类别: {DX_FILTER}")
print(f"处理总数: {len(records)}  合格: {accepted}  不合格: {rejected}")
print(f"总耗时: {total_time:.1f}s")
print(f"输出目录: {OUTPUT_DIR}")
print(f"  origin/   - {len(records)} 张")
print(f"  enhanced/ - {len(records)} 张")
print(f"  masks/    - {len(records)} 张")
print(f"  metadata.csv")
print(f"{'='*50}")

正在处理: 6705/6705

筛选类别: nv
处理总数: 6705  合格: 6663  不合格: 42
总耗时: 799.4s
输出目录: output/nv/
  origin/   - 6705 张
  enhanced/ - 6705 张
  masks/    - 6705 张
  metadata.csv


In [ ]:
# ============================================================
#                       主程序
# ============================================================

if __name__ == "__main__":
    path = 'data/ISIC_0027439.jpg'

    result = process_pipeline(path)

    # 输出状态
    status_text = "ACCEPTED" if result['is_valid'] else "REJECTED"
    print(f"\n{'='*40}")
    print(f"Image: {path}")
    print(f"Status: {status_text}")
    print(f"Hair ratio: {result['hair_ratio']:.4f}")
    print(f"Lesion Otsu threshold: {result['lesion_threshold']}")
    print(f"{'='*40}")

    # 显示结果
    cv2.imshow("Original image", result['original'])
    cv2.imshow("Hair mask", result['hair_mask'])
    cv2.imshow("After hair removal", result['hair_removed'])
    cv2.imshow("After CLAHE", result['enhanced'])
    cv2.imshow("Lesion mask", result['lesion_mask'])
    cv2.imshow("Lesion Overlay", result['overlay'])

    cv2.waitKey()
    cv2.destroyAllWindows()
    
    